In [1]:
# !pip install transformers torch pandas matplotlib peft

In [2]:
# !pip install flash-attn

In [1]:
import json
import math
import random
import torch



import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm


import matplotlib.pyplot as plt
import pandas as pd


In [2]:
from string import Template
prompt_q_without_contex_train= Template('''Instruct: $question
$options
$question
''')


prompt_without_contex_train= Template('''$question
Abbreviations: $abbreviation
          
Considering the following contexts:
context 1: $context1
context 2: $context2
context 3: $context3      
                                                                    
$question
$options
''')




cache_prompt= Template('''Instruct: $question
Abbreviations: $abbreviation
          
Considering the following contexts:
context 1: $context1
context 2: $context2
context 3: $context3      
                                                                    
$question
''')

option_prompt = Template('''
$options
Output: option ''')
# prompt_without_context = f'Hello {planet}'

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200")
model = AutoModelForSeq2SeqLM.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200", device_map="auto")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
def clean_question(question):
    for num in [14, 15, 16, 17, 18]:
        question = question.replace(f"[3GPP Release {num}]", "")
    return question


# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    return set(text.split())


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)
    
    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))
        
  
        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option
    if best_option == None:
        id =  random.randint(1, len(options))
        return options[id-1], id
    return best_option, options.index(best_option)+1

In [5]:
promptout = ''
answerout = ''

In [6]:
class MyLLMDataloader:
    def __init__(self, batch_size, tokenizer, data, shuffle = False, val= False):
        ## initializations
        self.batch_size  = batch_size
        self.tokenizer  = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        with open(data, "r") as f:
            self.data = json.load(f)
        self.all_examples = list(self.data.keys())
        self.shuffle = shuffle
        self.val = val
        
        self.n_data_points = math.ceil(len(self.data)/self.batch_size)
        self.indices = [i for i in range(self.n_data_points)]
        
    def __getitem__(self, idx):
        ## this gets a batch 
        global answerout
        global promptout
        option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 "]
        batch_start_id = idx * self.batch_size
        batch_end_id  = min(len(self.data), batch_start_id + self.batch_size) 
        batch = {"question_context":[], "answer":[]}
        
        for i in range(batch_start_id, batch_end_id):
            example = self.data[self.all_examples[i]]
            options = []
            for key in example.keys():
                if key.startswith("opt"):
                    options.append(example[key])


            correct_option_txt =example["answer"][10:]
            correct_option_id = int(example["answer"].split("option ")[1][0])  - 1
           

            if self.shuffle:
                random.shuffle(options)
                correct_option_id = options.index(correct_option_txt) +1
            
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                # correct_option_txt = example["answer"].split(": ")[1]
                correct_option_txt = example["answer"][10:]

                correct_option_txt_header = str(correct_option_id) +" " + correct_option_txt

                context1 = '\n'.join( [exx[:1000] for exx in example["context_qwen2"][:1]])
                # context2 = '\n'.join([exx[:50] for exx in example["context_gle"]])
                # context3 = '\n'.join([exx[:50] for exx in example["context_bm"]])

                context2 = ''
                context3 = ''

                prompt = prompt_without_contex_train.substitute(question = clean_question(example["question"]),\
                abbreviation='\n'.join(example["abbreviation"]), context1 = context1, context2 = context2, context3 = context3,
                options =options_with_header)
                promptout = prompt

                

            else:
                correct_option_id = correct_option_id+1
                correct_option_txt_header = str(correct_option_id) +" " + correct_option_txt
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                context1 = '\n'.join( [exx for exx in example["context_qwen2"][:1]])
                # context2 = '\n'.join([exx[:50] for exx in example["context_gle"]])
                # context3 = '\n'.join([exx[:50] for exx in example["context_bm"]])

                context2 = ''
                context3 = ''

                prompt = prompt_without_contex_train.substitute(question = clean_question(example["question"]),\
                abbreviation='\n'.join(example["abbreviation"]), context1 = context1, context2 = context2, context3 = context3,
                options =options_with_header)
                promptout = prompt
               

            ## context TBD
            # answer =  f"{correct_option_txt_header}  \nExplanation: {example['explanation']}"
            if not self.val:
                answer =  f"{correct_option_txt_header}  \nExplanation: {example['explanation']}"
            else:
                answer =  f"{correct_option_txt_header}"


            answerout = answer
       




            batch["question_context"] += [prompt]

            batch["answer"] += [answer]

        # self.tokenizer.padding_side = "left"
        q_tokens = self.tokenizer(batch["question_context"], padding="longest", max_length= 512, truncation=True, return_tensors="pt")  
        # self.tokenizer.padding_side = "right"
        a_tokens = self.tokenizer(batch["answer"], padding="longest", max_length= 128, truncation=True, return_tensors="pt")
        tokens = torch.cat([q_tokens["input_ids"], a_tokens["input_ids"]], dim=1)
        attn_masks = torch.cat([q_tokens["attention_mask"], a_tokens["attention_mask"]], dim=1)
        loss_mask = torch.cat([torch.zeros_like(q_tokens["attention_mask"]), a_tokens["attention_mask"]], dim=1)[:,1:]
        
        result = {
        "inp_ids":q_tokens["input_ids"],
        # "inp_mask":attn_masks[:,:-1],## Causal Training
        "inp_mask":q_tokens["attention_mask"],## Causal Training

        "out_ids":tokens[:,1:], ## Causal Labels
        "out_mask":a_tokens["attention_mask"],
        "q_tokens": q_tokens,
        # "a_tokens": a_tokens,
        "a_tokens": a_tokens,

        }
        # result["loss_mask"] = loss_mask * result["out_mask"]
        # result["out_ids"][:,:q_tokens["input_ids"].size(1)-10] = self.tokenizer.eos_token_id

        return result       


            

    def __iter__(self):
        self.idx = 0
        return self

    def __next__(self):
        if self.idx >= self.n_data_points:
            self.idx = 0
            raise StopIteration
        temp_idx = self.indices[self.idx]
        self.idx += 1
        return self[temp_idx]
                
    def __len__(self):
        return self.n_data_points
    
     

In [7]:
# with open("train5000_cleaned.json", "r") as f:
#     test_data = f.readlines()

# print(len(test_data))

In [8]:
def forward_pass(model, batch):
    inp_ids = batch["inp_ids"].to(model.device)
    attn_mask = batch["inp_mask"].to(model.device)

    a_tokens = batch['a_tokens']['input_ids'].to(model.device)
    print(inp_ids.shape, attn_mask.shape, a_tokens.shape)
    print(inp_ids[0][:10], attn_mask[0][:10], a_tokens[0][:10])
    a_tokens[a_tokens == tokenizer.pad_token_id] = -100
    # result = model(input_ids=inp_ids, attention_mask=attn_mask)


    result = model(input_ids=inp_ids,attention_mask =attn_mask, labels = a_tokens)
    logits = result.logits
    print(result.loss)
    return result.loss, logits


def calc_loss(loss_fn, logits, batch):
    B, L, C = logits.shape
    target = batch["out_ids"].to(logits.device)
    mask = batch["loss_mask"].to(logits.device)
    loss = loss_fn(logits.reshape(-1, C), target.reshape(-1)) * mask.reshape(-1)
    loss = loss.sum()/mask.sum()
    return loss

def update(model, optimizer, loss_fn, batch, accumulate_grad=True):
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss, logits = forward_pass(model, batch)
        # loss = calc_loss(loss_fn, logits, batch)
    
    scaler.scale(loss).backward()
    if not accumulate_grad:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    return loss.item(), logits




def train_on_batches(model, train_data_loader, optimizer, loss_fn, grad_accum_bs=16):
    train_loss = 0
    score = 0
    num_correct = 0
    total_num = 0
    mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5}

    pbar = tqdm(range(len(train_data_loader)))
    option_ids = [tokenizer(o).input_ids[0] for o in ["A", "B", "C", "D", "E"]]
    model.train()
    for i,batch in enumerate(train_data_loader):
        grad_step_bs = grad_accum_bs/train_data_loader.batch_size
        if (i>=grad_step_bs and i%grad_step_bs == 0) or i == len(train_data_loader)-1:
            accumulate_grad = False
        else:
            accumulate_grad = True
        loss, logits = update(model, optimizer, loss_fn, batch, accumulate_grad=accumulate_grad)
        train_loss += (1/(i+1))*(loss-train_loss)
        # if i% 50 == 0:
        #     print(logits.shape, logits[:,batch["q_tokens"].input_ids.size(1)-1,option_ids].shape, logits[ :,batch["q_tokens"].input_ids.size(1)-1:  ].shape, train_data_loader.tokenizer.batch_decode(logits[ :,batch["q_tokens"].input_ids.size(1)-1:  ].argmax(dim=2)))
        pred = (logits[:,batch["q_tokens"].input_ids.size(1)-1,option_ids].argmax(dim=1) + 1).tolist()

        target = train_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids[:,0], skip_special_tokens=True)
        for p,t in zip(pred, target):
            total_num += 1
            # print('p', p, 't',t)


            num_correct += (int(p) == int(t))
            score = num_correct/total_num
        

        pbar.set_description(f"Train Loss: {train_loss:.4f} Score: {score:.4f}")
        pbar.update(1)
    pbar.close()

    return train_loss, score



In [9]:
def train(model, tdataloader,vdataloader, optimizer, loss_fn, scheduler, epochs=3, log_dir="logs/realqwen3B_noQ"):
    os.makedirs(log_dir, exist_ok=True)
    with open(f"{log_dir}/train.txt", "w") as tf, open(f"{log_dir}/val.txt", "w") as vf:
        tf.write(f"Epoch,Loss,Score\n")
        vf.write("Epoch,Loss,Score\n")
    best_val_score = 0
    
    for epoch in range(1,1+epochs):
        train_loss, tscore = train_on_batches(model, tdataloader, optimizer, loss_fn, grad_accum_bs=16)
        scheduler.step()
        val_loss, score = validation(model, vdataloader, loss_fn)
        with open(f"{log_dir}/train.txt", "a+") as tf, open(f"{log_dir}/val.txt", "a+") as vf:
            tf.write(f"{epoch},{train_loss},{tscore}\n")
            vf.write(f"{epoch},{val_loss},{score}\n")
        lr = optimizer.param_groups[0]["lr"]
        tqdm.write(f"Epoch: {epoch} | LR: {lr:.7f} | Train Loss: {train_loss:.4f} | Train Score: {tscore:.4f} | Val Loss: {val_loss:.4f} | Val Score: {score:.4f}")
        
        if score > best_val_score:
            model.save_pretrained(log_dir+"/model")
            best_val_score = score

        # update the learning rate
        # for param_group in optimizer.param_groups:
        #     param_group['lr'] = 1.0*float(param_group['lr'])

        train_df = pd.read_csv(f"{log_dir}/train.txt")
        val_df = pd.read_csv(f"{log_dir}/val.txt")

        fig, ax = plt.subplots(1, 2, figsize=(20,8))
        ax[0].plot(range(1,len(train_df)+1), train_df["Loss"], label="Train")
        ax[0].plot(range(1,len(val_df)+1), val_df["Loss"], label="Val")

        ax[1].plot(range(1,len(train_df)+1), train_df["Score"], label="Train")
        ax[1].plot(range(1,len(val_df)+1), val_df["Score"], label="Val")

        ax[0].set_xlabel("Epochs")
        ax[1].set_xlabel("Epochs")

        ax[0].set_ylabel("Loss")
        ax[1].set_ylabel("Score")

        ax[0].legend()
        ax[1].legend()

        fig.suptitle('', fontsize=16)
        fig.savefig("plt.png")


def validation(model, val_data_loader, loss_fn):
    val_loss = 0
    mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5}

    score = 0
    num_correct = 0
    pbar = tqdm(range(len(val_data_loader)))
    option_ids = [tokenizer(o).input_ids[0] for o in ["A", "B", "C", "D", "E"]]
    total_num = 0
    model.eval()
    with torch.inference_mode():
        for i, batch in enumerate(val_data_loader):
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = forward_pass(model, batch)
                loss = calc_loss(loss_fn, logits, batch).item()
            val_loss += (1/(i+1))*(loss - val_loss)
            pred = (logits[:,batch["q_tokens"].input_ids.size(1)-1,option_ids].argmax(dim=1) + 1).tolist()
            target = val_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids[:,0], skip_special_tokens=True)
            for p,t in zip(pred, target):
             
                total_num += 1
                t = mapper_ans[t]
                num_correct += (int(p) == int(t))
                score = num_correct/total_num
                
            pbar.set_description(f"Val Loss: {val_loss:.4f} Score: {score:.4f}")
            pbar.update(1)
            
        gen_tokens = model.generate(**batch["q_tokens"].to(model.device), max_new_tokens=10)[:,batch["q_tokens"].input_ids.size(1):]
        gen_txt = val_data_loader.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        target_txt = val_data_loader.tokenizer.batch_decode(batch["a_tokens"].input_ids, skip_special_tokens=True)
        tqdm.write("Target: " + "\n" + "\n".join(target_txt) + "\nGenerated: \n " + "\n".join(gen_txt) +"\n")
        pbar.close()

    return val_loss, score


In [10]:
# device = "cuda"
# torch.set_default_device(device)
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, device_map ="auto")
epochs = 7
lr = 1e-4

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.eos_token_id, reduction='none').cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay= 0.001)
optimizer.zero_grad()


# trainLoader = TrainDataLoader(2, tokenizer, topk=topk)
trainLoader = MyLLMDataloader(1, tokenizer, "cleaned_TeleQnA_train_context_gte.json", val=False, shuffle=True)

valLoader = MyLLMDataloader(1, tokenizer, "questions_365_val.json", val=True)
scaler =  torch.cuda.amp.GradScaler(enabled=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
            T_max = epochs*2, eta_min=1E-8)
train(model, trainLoader, valLoader,optimizer, loss_fn, scheduler, epochs=10)



FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_TeleQnA_train_context_gte.json'

In [20]:
promptout


'What is the purpose of the Nmfaf_3daDataManagement_Deconfigure service operation? \nAbbreviations: \n          \nConsidering the following contexts:\ncontext 1: 4.2.2\tService Operations\nIf the MFAF determines the received HTTP PUT request needs to be redirected, the MFAF shall send an HTTP redirect response as specified in clause\xa06.10.9 of 3GPP\xa0TS\xa029.500\xa0[4].\n4.2.2.3\tNmfaf_3daDataManagement_Deconfigure service operation\n4.2.2.3.1\tGeneral\nThe Nmfaf_3daDataManagement_Deconfigure service operation is used by an NF service consumer to stop mapping data or analytics received by the MFAF to one or more out-bound notification endpoints.\n4.2.2.3.2\tStop mapping data or analytics \nFigure\xa04.2.2.3.2-1 shows a scenario where the NF service consumer sends a request to the MFAF to update the configuration to stop mapping data or analytics (as shown in 3GPP\xa0TS\xa023.288\xa0[14])\nFigure\xa04.2.2.3.2-1: NF service consumer stops mapping data or analytics\nThe NF service con

In [21]:
# the following 2 hyperparameters are task-specific
max_source_length = 512
max_target_length = 128

input_sequences = [promptout]

encoding = tokenizer(
    [sequence for sequence in input_sequences],
    padding="longest",
    max_length=max_source_length,
    truncation=True,
    return_tensors="pt",
)

input_ids, attention_mask = encoding.input_ids, encoding.attention_mask
# encode the targets
target_encoding = tokenizer(
    [answerout],
    padding="longest",
    max_length=max_target_length,
    truncation=True,
    return_tensors="pt",
)
labels = target_encoding.input_ids

# replace padding token id's of the labels by -100 so it's ignored by the loss
labels[labels == tokenizer.pad_token_id] = -100

print(input_ids.shape,attention_mask.shape, labels.shape)
print(input_ids[0][:10],attention_mask[0][:10], labels[0][:10])

# forward pass
loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
loss.item()

torch.Size([1, 432]) torch.Size([1, 432]) torch.Size([1, 76])
tensor([ 363,   19,    8, 1730,   13,    8,  445,   51,   89,    9]) tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]) tensor([  314,   304, 16302,     8,     3, 13286,  6282,    12,  1190, 14670])


0.713148295879364

In [22]:
tokenizer.pad_token_id

1

In [65]:
# with open('cleaned_TeleQnA_train_context_gte.json', 'r') as f:
#     new_train = f.read()

In [ ]:
# with open('cleaned_TeleQnA_train_context_gte.json', 'r') as f:
#     new_train = f.read()

# new_train = json.loads(new_train)


# with open('TeleQnA_training.json', 'r') as f:
#     old_train = f.read()

# old_train = json.loads(old_train)

# with open('cleaned_TeleQnA_train_context_gte.json', 'w') as f:
#     json.dump(new_train, f)

# for ex in old_train:
#     new_train[ex]['answer'] = old_train[ex]['answer']
    